<a href="https://colab.research.google.com/github/Anmolcode234/Movie-Recommender-System/blob/main/CREDIT_CARD_FRAUD_DETECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Credit card Fraud Detection - Logistic Regression
# Leakage free pipeline (PCA features + Time/Amount preprocessing)
#===========================================================
#-----------------------------------
# Import Libraries
#-----------------------------------

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

In [7]:
df = pd.read_csv("/content/creditcard.csv.zip")
print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

    

In [8]:
# Target Imabalnce Check

print("\nClass Distribution (%):")
print(df["Class"].value_counts(normalize=True) * 100)





Class Distribution (%):
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64


In [9]:
# Basic Data Cleaning
print("\nBasic info:")
df.info()

df = df.drop_duplicates()
print("\nShape after dropping duplicates:", df.shape)


Basic info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 2

In [12]:
# Separate Features and Target
X = df.drop(columns="Class")
y = df["Class"]

In [13]:
# Train-Test Split
x_train, x_test, y_train, y_test = train_test_split(
   X,
   y,
   test_size=0.2,
   random_state=42,
   stratify=y
)
print(f"\nTrain shape: {x_train.shape}, Test shape: {x_test.shape}")
print(f"Train fraud rate: {y_train.mean()* 100:.4f}%")
print(f"Test fraud rate: {y_test.mean()* 100:.4f}%")



Train shape: (226980, 30), Test shape: (56746, 30)
Train fraud rate: 0.1665%
Test fraud rate: 0.1674%


In [14]:
# Separate PCA and NON-PCA Features

pca_features = x_train.iloc[:, 1:29]
non_pca_features = x_train[["Time", "Amount"]]


In [15]:
# Diagnostic Analysis- PCA Features
diagnosis_pca = {}
for coulmn in pca_features.columns:
    diagnosis_pca[coulmn] = {
        "Missing Values": pca_features[coulmn].isnull().sum(),
        "Variance": pca_features[coulmn].var(),
        "Standard Deviation": pca_features[coulmn].std(),
        "Range": (pca_features[coulmn].min(), pca_features[coulmn].max()),
        "Infinite Values": (np.isinf(pca_features[coulmn]).sum()),
    }

diagnosis_pca = pd.DataFrame.from_dict(diagnosis_pca, orient="index")
print("\nPCA Features Diagnostic Analysis:")
print(diagnosis_pca)




PCA Features Diagnostic Analysis:
     Missing Values  Variance  Standard Deviation  \
V1                0  3.791951            1.947293   
V2                0  2.721236            1.649617   
V3                0  2.271863            1.507270   
V4                0  2.007776            1.416960   
V5                0  1.848628            1.359643   
V6                0  1.751389            1.323401   
V7                0  1.440434            1.200181   
V8                0  1.367362            1.169342   
V9                0  1.196956            1.094055   
V10               0  1.151784            1.073212   
V11               0  1.037672            1.018662   
V12               0  0.984790            0.992366   
V13               0  0.988071            0.994018   
V14               0  0.910667            0.954289   
V15               0  0.835925            0.914289   
V16               0  0.763287            0.873663   
V17               0  0.710844            0.843116   
V18        

In [16]:
# Diagnosis Analysis Non-PCA Features (Time & Amount)

diagnosis_non_pca = {}
for coulmn in non_pca_features.columns:
  Q1 = non_pca_features[coulmn].quantile(0.25)
  Q3 = non_pca_features[coulmn].quantile(0.75)
  IQR = Q3 - Q1
  Lower_Bound = Q1 - 1.5 * IQR
  Upper_Bound = Q3 + 1.5 * IQR
  outliers = non_pca_features[(non_pca_features[coulmn] < Lower_Bound) | (non_pca_features[coulmn] > Upper_Bound)].sum()

  diagnosis_non_pca[coulmn] = {
      "Missing Values": non_pca_features[coulmn].isnull().sum(),
      "Variance": non_pca_features[coulmn].var(),
      "Standard Deviation": non_pca_features[coulmn].std(),
      "Range": (non_pca_features[coulmn].min(), non_pca_features[coulmn].max()),
      "Infinite Values": (np.isinf(non_pca_features[coulmn]).sum()),
      "Outliers": outliers,
      "Skewness": non_pca_features[coulmn].skew(),

  }

diagnosis_non_pca = pd.DataFrame.from_dict(diagnosis_non_pca, orient="index")
print("\nNon-PCA Features Diagnostic Analysis:")
print(diagnosis_non_pca)



Non-PCA Features Diagnostic Analysis:
        Missing Values      Variance  Standard Deviation            Range  \
Time                 0  2.255130e+09        47488.214282  (0.0, 172792.0)   
Amount               0  6.040144e+04          245.767046  (0.0, 19656.53)   

        Infinite Values                                           Outliers  \
Time                  0         Time      0.0
Amount    0.0
dtype: float64   
Amount                0  Time      2.365293e+09
Amount    1.295949e+07
...   

         Skewness  
Time    -0.037474  
Amount  14.393212  


In [18]:
# Custom Transformer - IQR Outlier Clipper

class IQROutlierClipper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.q1_ = np.percentile(X, 25, axis=0)
        self.q3_ = np.percentile(X, 75, axis=0)
        self.iqr_ = self.q3_ - self.q1_
        return self

    def transform(self, X):
        lower = self.q1_ - 1.5 * self.iqr_
        upper = self.q3_ + 1.5 * self.iqr_
        return np.clip(X, lower, upper)

In [21]:
# Build Preprocessing Pipeline

non_pca_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("power_transformer", PowerTransformer(method="yeo-johnson")),
    ("outlier_clipper", IQROutlierClipper())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("non_pca", non_pca_pipeline, ["Time", "Amount"]),
    ],
    remainder="passthrough",
)

In [22]:
# Full Modeling Pipeline
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression())
])
#


In [24]:

# Hyprometer Grid

param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10,],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"],
    "model__class_weight": ["balanced"],
    "model__max_iter": [500]
}


In [25]:
# Startified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [27]:
# Hyperparameter Tunning (Optimice Avreage Precision / PR-AUC)
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)

print("\nStarting Grid Searchcv...")
grid_search.fit(x_train, y_train)

print("\nBest Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation Average Precision (PR-AUC):")
print(f"{grid_search.best_score_:.6f}")


Starting Grid Searchcv...
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Best Parameters:
{'model__C': 1, 'model__class_weight': 'balanced', 'model__max_iter': 500, 'model__penalty': 'l2', 'model__solver': 'liblinear'}

Best Cross-Validation Average Precision (PR-AUC):
0.755975


In [28]:
# Predictions on Unseen Test Set
y_pred = grid_search.predict(x_test)
y_prob = grid_search.predict_proba(x_test)[:, 1]


In [30]:
# Model Evaluation
print("\n" + "=" * 60)
print("MODEL PERFORMANCE ON TEST SET")
print("=" * 60)

print(f"Accuracy            :{accuracy_score(y_test, y_pred):.4f}")
print(f"Balanced Accuracy   :{balanced_accuracy_score(y_test, y_pred):.4f}")
print(f"Precision           :{precision_score(y_test, y_pred):.4f}")
print(f"Recall              :{recall_score(y_test, y_pred):.4f}")
print(f"F1 Score            :{f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC             :{roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC              :{grid_search.best_score_:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))


MODEL PERFORMANCE ON TEST SET
Accuracy            :0.9749
Balanced Accuracy   :0.9244
Precision           :0.0555
Recall              :0.8737
F1 Score            :0.1044
ROC AUC             :0.9621
PR-AUC              :0.7560

Confusion Matrix:
[[55239  1412]
 [   12    83]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9998    0.9751    0.9873     56651
           1     0.0555    0.8737    0.1044        95

    accuracy                         0.9749     56746
   macro avg     0.5277    0.9244    0.5458     56746
weighted avg     0.9982    0.9749    0.9858     56746

